# 01 — Data Exploration
Explore the raw fabric defect dataset: class distribution, image statistics, and augmentation previews.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

DATA_DIR = Path('../data/raw')

## Class Distribution

In [ ]:
class_counts = {}
for cls_dir in sorted(DATA_DIR.iterdir()):
    if cls_dir.is_dir():
        count = len(list(cls_dir.glob('*.[jpJP][pnPN][gG]*')))
        class_counts[cls_dir.name] = count

df = pd.DataFrame(list(class_counts.items()), columns=['Defect Class', 'Image Count'])
print(df.to_string(index=False))
print(f"\nTotal images: {df['Image Count'].sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=df.sort_values('Image Count'), x='Image Count', y='Defect Class', palette='viridis', ax=ax)
ax.set_title('Defect Class Distribution')
plt.tight_layout()
plt.show()

## Sample Images per Class

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (cls, _) in zip(axes.flatten(), class_counts.items()):
    imgs = list((DATA_DIR / cls).glob('*.[jpJP][pnPN][gG]*'))
    if imgs:
        img = cv2.imread(str(imgs[0]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
    ax.set_title(cls.replace('_', ' ').title(), fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Images — One per Defect Class', fontsize=13)
plt.tight_layout()
plt.show()

## Augmentation Preview

In [ ]:
import albumentations as A
from src.detection.dataset import get_train_transforms

sample_imgs = [str(p) for cls_dir in DATA_DIR.iterdir() if cls_dir.is_dir()
               for p in list(cls_dir.glob('*.[jpJP][pnPN][gG]*'))[:1]]

if sample_imgs:
    transform = get_train_transforms(380)
    img = cv2.cvtColor(cv2.imread(sample_imgs[0]), cv2.COLOR_BGR2RGB)
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
    for i in range(1, 5):
        aug = transform(image=img)['image'].permute(1, 2, 0).numpy()
        aug = (aug * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])).clip(0, 1)
        axes[i].imshow(aug); axes[i].set_title(f'Aug {i}'); axes[i].axis('off')
    plt.suptitle('Tropical Augmentation Preview')
    plt.tight_layout()
    plt.show()